# Plan de Trabajo — Proyecto Final: Predicción de Churn (Interconnect)

**Etapa 1 del Proyecto Final del Bootcamp de Data Science (TripleTen)**  
*Documento preliminar para revisión del líder de equipo antes de comenzar el desarrollo del modelo.*

## Contexto del caso

El operador de telecomunicaciones **Interconnect** quiere saber qué clientes van a cancelar sus servicios para ofrecerles códigos promocionales y planes especiales de retención. El equipo de marketing ha facilitado cuatro archivos con información del contrato, datos personales, servicios de internet y servicios de telefonía, todos unibles por `customerID`. La información tiene un rango de fechas a partir del **1 de octubre de 2013** hasta el **1 de febrero de 2020**, rango de fechas que se usa como *fotografía* para definir el objetivo.

- **Target:** `Churn = 1` cuando `EndDate != 'No'` (el cliente ya canceló).
- **Métrica principal:** AUC-ROC. **Métrica adicional:** exactitud.
- **Criterio de aprobación:** AUC-ROC ≥ 0.75 (mínimo) · ≥ 0.88 (puntuación máxima).
- **Fuentes de datos:** `contract.csv`, `personal.csv`, `internet.csv`, `phone.csv`.

## 1. Entendimiento del problema

Interconnect enfrenta un problema de **clasificación binaria supervisada** para predecir, a partir de los atributos conocidos de cada cliente (contrato, demografía, servicios contratados y consumo), la probabilidad de que un clinete cancele su contrato en un corto  plazo. El valor de negocio no está solo en clasificar con precisión, sino en **rankear correctamente** — por eso la métrica principal es AUC-ROC y no exactitud, ya que las clases están desbalanceadas y el uso del modelo será priorizar una lista de clientes a contactar o extender su contrato con promociones.

## 2. Preguntas aclaratorias al líder de equipo

**Sobre negocio y alcance**

- ¿Hay un presupuesto máximo de clientes contactables por campaña de retención, o podemos ofrecer promoción a todos los clasificados como "alto riesgo"?

**Sobre los datos**

- Los archivos `internet.csv` (5 518 filas) y `phone.csv` (6 362 filas) tienen menos registros que `contract.csv` (7 043). ¿Se debe interpretar la ausencia de un cliente en `internet.csv` como "no tiene internet contratado", o como dato faltante que debe imputarse?
- ¿Hay restricciones sobre qué features se pueden usar? En particular, `EndDate` y `TotalCharges` pueden producir ***data leakage*** si no se tratan con cuidado (el primero *es* el target, y el segundo correlaciona con la antigüedad).
- ¿El entregable será una lista, un CSV o se mandará a una base de datos al equipo de marketing?

**Sobre el modelado**

- ¿Se permite usar modelos de *gradient boosting* externos (LightGBM / CatBoost / XGBoost) y *redes neuronales* (TensorFlow / PyTorch), o se requiere limitarse a Scikit-learn estándar (Modelos clasicos) ?
- ¿Hay restricciones de **tiempo de entrenamiento** o de **tiempo de inferencia**? Por ejemplo, ¿el modelo debe poder reentrenarse diariamente sobre datos nuevos, o evaluarse en tiempo real sobre el CRM? Este punto puede descartar modelos muy pesados aunque den mejor AUC-ROC.
- ¿Dónde se realizará el despliegue del modelo, se pondrá en producción o será de uso interno local?

## 3. Plan de trabajo

1. **Carga, auditoría de calidad y unificación.** Cargar los 4 CSV, auditar tipos, nulos y duplicados, y unirlos por `customerID` mediante *left join* desde `contract.csv` (la tabla con cobertura completa). Justificación: `internet` y `phone` tienen cobertura parcial, y el *left join* preserva a todos los clientes permitiendo codificar la ausencia como "no tiene ese servicio contratado".

2. **Definición del target y *feature engineering*.** Construir la variable `Churn` a partir de `EndDate`, crear `days_active` (diferencia entre `BeginDate` y la fecha de corte 2020-02-01), contar el número total de servicios contratados por cliente y calcular métricas derivadas como *LTV* (Customer Lifetime Value, `MonthlyCharges × meses_activo`). Convertir `TotalCharges` a numérico y resolver sus nulos (que corresponden a clientes muy nuevos).

3. **EDA orientado al churn.** Comparar la distribución de cada feature entre clientes que se van y los que se quedan, con foco en tipo de contrato, método de pago, servicios contratados y antigüedad. Con el onbjetivo de identificar los *drivers* más fuertes del churn, detectar posibles fuentes de *fuga de datos* y documentar hallazgos visuales que luego se reutilizarán en el informe final.

4. **Modelado comparativo.** Separar *train/test* estratificado y entrenar al menos tres modelos: *Logistic Regression* como baseline interpretable, *Random Forest*, y un *gradient boosting* (LightGBM o CatBoost). Validación mediante *cross-validation* estratificada con AUC-ROC como *scoring*. Tunear hiperparámetros del mejor candidato con `GridSearchCV` y manejar el desbalance con `class_weight='balanced'` o `scale_pos_weight` e implementar, en caso necesario, un sobremuestreo para balancear las clases.

5. **Evaluación final e interpretación.** Evaluar el modelo ganador sobre el set de *test* reservado, reportando AUC-ROC, exactitud y matriz de confusión en una tabla comparativa. Analizar importancia de *features* (SHAP o *feature importance* nativa) para interpretabilidad, y **traducir los hallazgos en recomendaciones accionables de retención** para el equipo de marketing.

## 4. Criterio de éxito

Alcanzar **AUC-ROC ≥ 0.88** en el *test set* para obtener la puntuación máxima del sprint (6 SP). El umbral mínimo aceptable es 0.75, y el objetivo realista de este plan es posicionarse en el rango **0.85–0.88**, que ya se considera un modelo sólido y operable en producción.

## 5. Respuestas acordadas con el líder de equipo

Tras la revisión inicial quedaron cerrados los siguientes puntos, que se aplicaron durante el desarrollo del cuaderno principal.

| Pregunta | Respuesta aplicada |
|----------|--------------------|
| Ausencia de un cliente en `internet.csv` / `phone.csv` | Se interpreta como **servicio no contratado**, no como dato faltante. Se codifica como *Sin servicio* o `False`. |
| Uso de `EndDate` como variable | **No se usa como predictor** (es directamente el target). Solo se usa para derivar `Churn` y `DaysActive`. |
| Uso de `TotalCharges` | Se **descarta para modelado** por multicolinealidad con `DaysActive` y `MonthlyCharges`. |
| Métrica principal | **AUC-ROC**, con umbral mínimo aceptable 0.75 y objetivo 0.88. |
| Desbalance de clases (73.5 / 26.5) | Se compensa con `class_weight='balanced'` o `scale_pos_weight`. |
| Línea base | **DummyClassifier** (estrategia *stratified*) como referencia mínima. |

## 6. Iteraciones y feedback del líder de equipo

Esta sección recoge las notas de revisión posteriores al plan original. Se mantiene separada para dejar claro qué proviene del documento inicial y qué corresponde al ciclo iterativo.

<div class="alert alert-block alert-success">
<b>Comentario general (1ra Iteracion)</b> <a class=“tocSkip”></a>

Hola, Fernando. Excelente explicación sobre tu plan de trabajo para tu proyecto final! Muy bien ilustrado y explicado solamente te recomiendo el uso de diagramas.

Agregas un apartado de preparación y limpieza de datos para fortalecer nuestro análisis. En este punto ajustas el tipo de variables y juntar las bases. Como aprendiste en cursos pasados, el análisis de valores nulos y duplicados es muy importante para disminuir sesgos en nuestros resultados. 


Colocas un  Análisis Exploratorio de Datos (EDA) completo, puedes incluir gráficas representativas y conclusiones relevantes para cada una. Recuerda que es importante verificar el balance de clases en la variable objetivo, que lo colocas en un paso posterior. 


Agregas la consideración el análisis de modelos. Solamente te recomiendo colocar la matriz de correlación para identificar posibles colinealidades. 

Puedes complementar con el ajuste de maneja de colinealidad y uso de RandomizedSearchCV para encontrar de manera eficiente los hiperparamentros que mejor se adaptan a tu modelo. 

Para lo siguiente, mencionas métricas (por ejemplo, F1-score, accuracy, etc.) son altas en el conjunto de entrenamiento pero bajas en el de prueba, esto indica un posible sobreajuste (overfitting). En tal caso, será necesario ajustar el proceso de entrenamiento para reducir este problema. Usas correcto el AU-ROC. 

Como referencia, puedes utilizar un umbral de 0.88 en el F1-score para considerar que el modelo tiene un desempeño adecuado. También se recomienda realizar una prueba de cordura utilizando un DummyClassifier como modelo base. Finalmente, asegúrate de identificar correctamente si el problema que estás abordando es de regresión o de clasificación, ya que esto determinará la elección del modelo y las métricas de evaluación adecuadas.

### 6.1 Respuestas a los comentarios de la primera iteración

A continuacion, cómo se aplicaron los puntos de feedback del líder de equipo:

| Comentario del líder | Acción tomada en el cuaderno principal |
|----------------------|----------------------------------------|
| "Agrega preparación y limpieza de datos" | Sección 3 (Auditoría por dataset) + **auditoría de nulos consolidada post-merge** con tabla de razón probable por columna. |
| "Agrega EDA completo con gráficas y conclusiones" | Secciones 4.1 a 5.3: boxplots de antigüedad, histogramas por tipo de contrato, evolución por año, distribucion del LTV. |
| "Verifica el balance de clases" | Proporciones (73.5 / 26.5) + estrategia con **upsampling** solo en train + CV sobre train crudo. |
| "Matriz de correlacion para identificar colinealidades" | Matriz 16x16 + diagnóstico **VIF** + exclusión de `TotalCharges` y `LTV`. |
| "Uso de RandomizedSearchCV" | **Aplicado a cinco modelos** (DT, LogReg, RF, LightGBM, XGBoost) con **40 iteraciones cada uno** (~1,000 fits en total, 5-fold CV interno). Mejora de +0.015 AUC respecto a defaults. |
| "Cuidar overfitting" | Diferencia CV vs test <= 0.02 en todos los modelos. Tabla explicita **"Con CV vs Sin CV"** al final. |
| "Referencia F1 y DummyClassifier" | DummyClassifier como baseline (AUC = 0.50). Tabla comparativa con Accuracy, F1, APS y AUC-ROC. |
| "Regresión vs clasificación" | Clasificación binaria; métrica principal AUC-ROC. |

### 6.2 Cierre de la segunda iteración (modelo entrenado)

**Entregables de la fase 2:**

- **12 configuraciones de modelos** evaluadas: siete base (Dummy + DecisionTree + LogReg + RF + LGBM + XGB + MLP), cinco tuneados con grid amplio, y un ensemble de los tres mejores.
- **Red neuronal**: `MLPClassifier` de scikit-learn con arquitectura `128 -> 64`, early stopping interno y regularización L2 (`alpha=1e-3`). Se simplificó la versión anterior (Keras + CV manual) para mantener la misma API que el resto de los modelos.
- **Tabla comparativa** con AUC test, CV AUC y tiempos.
- **Tabla explicita de métricas con y sin CV** para confirmar estabilidad (diferencias <= 0.02).
- **Tabla de importancia de variables** por consenso de seis modelos.
- **Feature engineering**: `AddonsCount` (número de servicios de valor agregado) y `FiberNoSecurity` (interacción clasica).
- **Función `exportar_lista_riesgo`**: genera CSV + Excel para marketing con 5,174 clientes activos, segmentados por percentiles (Alto 15 %, Medio 25 %, Bajo 60 %).
- **Retrospectiva técnica** al final con los 6 problemas enfrentados y sus soluciones, incluida la **fuga por construcción en `DaysActive`**, corregida con la variable alternativa `Tenure`.

**Resultado final:**

| Métrica              | Valor |
|----------------------|-------|
| AUC-ROC objetivo     | 0.88  |
| **AUC-ROC obtenido** | **0.851** (LightGBM tuneado, test) |
| AUC-ROC CV (mejor)   | 0.860 (ensemble top 3) |
| AUC-ROC mínimo       | 0.75  (superado) |
| Modelo en producción | LightGBM tuneado (velocidad + precisión) |
| Clientes activos     | 5,174 (en lista de retención) |
| Alto riesgo          | 776 (15 %) |

El modelo esta **por encima del mínimo aceptable y ~0.03 por debajo del objetivo ambicioso**. Se exploraron todas las palancas del dataset actual:

1. Varios algoritmos (lineal, arbol, ensemble, boosting, red neuronal).
2. Tuning con `RandomizedSearchCV` sobre XGBoost (+0.02 AUC respecto al base).
3. Feature engineering (`AddonsCount`, `FiberNoSecurity`).
4. Ensemble promedio de los tres mejores (iguala al mejor individual en test y lo supera en CV).
5. Red neuronal (MLP con early stopping) como referencia no lineal.

El techo realista (~0.86 CV) se alcanza con las fuentes actuales. Cerrar la brecha de 0.03 al objetivo ambicioso requiere fuentes externas: consumo mensual de datos, historial de quejas y contactos con soporte técnico.